# DSCI 552 - Homework 3

*Name:* Nithya Prasasthani Eeri  
*GitHub Username:* Nithya0605-eeri  
*USC ID:* 9182042369

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### 1(a) AReM Dataset

The AReM dataset was downloaded from the UCI Machine Learning Repository. It contains seven activity folders: bending1, bending2, cycling, lying, sitting, standing, and walking. Each instance contains six time series, and each time series contains 480 consecutive values.

### 1(b) Training and Test Data

For bending1 and bending2, datasets 1 and 2 are used as the test set. For cycling, lying, sitting, standing, and walking, datasets 1, 2, and 3 are used as the test set. All remaining datasets are used as the training set.

In [4]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter(action="ignore", category=FutureWarning)

np.random.seed(42)

In [5]:
data_path = "Downloads/Eeri_NithyaPrasasthani_HW3/data/AReM"

activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]



In [6]:
train_files = []
test_files = []

for activity in activities:
    folder_path = os.path.join(data_path, activity)
    files = sorted(
        [f for f in os.listdir(folder_path) if f.endswith(".csv")],
        key=lambda x: int(x.replace("dataset", "").replace(".csv", ""))
    )

    for file in files:
        dataset_num = int(file.replace("dataset", "").replace(".csv", ""))

        # Test set:
        # bending1 and bending2 -> datasets 1 and 2
        # all other activities -> datasets 1, 2, and 3
        if activity in ["bending1", "bending2"]:
            is_test = dataset_num in [1, 2]
        else:
            is_test = dataset_num in [1, 2, 3]

        full_path = os.path.join(folder_path, file)

        if is_test:
            test_files.append((full_path, activity))
        else:
            train_files.append((full_path, activity))

print("Number of training instances:", len(train_files))
print("Number of test instances:", len(test_files))

print("\nFirst 5 training files:")
for item in train_files[:5]:
    print(item)

print("\nFirst 5 test files:")
for item in test_files[:5]:
    print(item)

Number of training instances: 69
Number of test instances: 19

First 5 training files:
('Downloads/Eeri_NithyaPrasasthani_HW3/data/AReM/bending1/dataset3.csv', 'bending1')
('Downloads/Eeri_NithyaPrasasthani_HW3/data/AReM/bending1/dataset4.csv', 'bending1')
('Downloads/Eeri_NithyaPrasasthani_HW3/data/AReM/bending1/dataset5.csv', 'bending1')
('Downloads/Eeri_NithyaPrasasthani_HW3/data/AReM/bending1/dataset6.csv', 'bending1')
('Downloads/Eeri_NithyaPrasasthani_HW3/data/AReM/bending1/dataset7.csv', 'bending1')

First 5 test files:
('Downloads/Eeri_NithyaPrasasthani_HW3/data/AReM/bending1/dataset1.csv', 'bending1')
('Downloads/Eeri_NithyaPrasasthani_HW3/data/AReM/bending1/dataset2.csv', 'bending1')
('Downloads/Eeri_NithyaPrasasthani_HW3/data/AReM/bending2/dataset1.csv', 'bending2')
('Downloads/Eeri_NithyaPrasasthani_HW3/data/AReM/bending2/dataset2.csv', 'bending2')
('Downloads/Eeri_NithyaPrasasthani_HW3/data/AReM/cycling/dataset1.csv', 'cycling')


### 1(c)(i) Common Time-Domain Features

Time-domain features summarize the statistical and temporal characteristics of a time series directly from its observed values. Common features used in time-series classification include:

- Minimum
- Maximum
- Mean
- Median
- Standard deviation
- Variance
- First quartile (Q1)
- Third quartile (Q3)
- Range
- Interquartile range (IQR)
- Mean absolute deviation
- Skewness
- Kurtosis
- Autocorrelation
- Trend or slope
- Number of peaks
- Zero-crossing rate

These features describe properties such as the central tendency, spread, distribution, temporal dependence, and changes in the signal. Feature-based time-series classification commonly uses statistical time-domain features such as mean, maximum, minimum, standard deviation, skewness, and kurtosis, along with structural characteristics such as trend and periodicity. [Source: ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S1568494622005889)

### 1(c)(ii) Time-Domain Feature Extraction

For each of the 88 instances, the following seven time-domain features are extracted from each of the six time series: minimum, maximum, mean, median, standard deviation, first quartile, and third quartile. This results in 42 features for each instance.

In [9]:
# Function to read and clean an AReM dataset file

def read_arem_file(file_path):
    df = pd.read_csv(
        file_path,
        skiprows=5,
        header=None,
        usecols=range(7),
        names=[
            "time",
            "avg_rss12",
            "var_rss12",
            "avg_rss13",
            "var_rss13",
            "avg_rss23",
            "ar_rss23"
        ],
        engine="python"
    )

    sensor_cols = [
        "avg_rss12",
        "var_rss12",
        "avg_rss13",
        "var_rss13",
        "avg_rss23",
        "ar_rss23"
    ]

    for col in sensor_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=sensor_cols)

    return df


# Test the cleaned function
sample_df = read_arem_file(
    os.path.join(data_path, "bending1", "dataset1.csv")
)

print("Shape:", sample_df.shape)
sample_df.head()

Shape: (480, 7)


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,ar_rss23
0,0,39.25,0.43,22.75,0.43,33.75,1.3
1,250,39.25,0.43,23.00,0.00,33.00,0.0
2,500,39.25,0.43,23.25,0.43,33.00,0.0
3,750,39.50,0.50,23.00,0.71,33.00,0.0
4,1000,39.50,0.50,24.00,0.00,33.00,0.0


In [10]:
# Extract the 7 required time-domain features from each of the 6 time series

sensor_cols = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "ar_rss23"
]

def extract_features(df):
    features = {}

    for i, col in enumerate(sensor_cols, start=1):
        x = df[col]

        features[f"min{i}"] = x.min()
        features[f"max{i}"] = x.max()
        features[f"mean{i}"] = x.mean()
        features[f"median{i}"] = x.median()
        features[f"std{i}"] = x.std()
        features[f"1st_quart{i}"] = x.quantile(0.25)
        features[f"3rd_quart{i}"] = x.quantile(0.75)

    return features


# Test feature extraction on one instance
sample_features = extract_features(sample_df)

pd.DataFrame([sample_features])

,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,37.25,45.0,40.624792,40.5,1.476967,39.25,42.0,0.0,1.3,0.358604,...,2.188449,33.0,36.0,0.0,1.92,0.570583,0.43,0.582915,0.0,1.3


In [11]:
# Extract features from all 88 instances

all_features = []

for activity in activities:
    folder_path = os.path.join(data_path, activity)

    files = sorted(
        [f for f in os.listdir(folder_path) if f.endswith(".csv")],
        key=lambda x: int(x.replace("dataset", "").replace(".csv", ""))
    )

    for file in files:
        file_path = os.path.join(folder_path, file)

        df = read_arem_file(file_path)
        features = extract_features(df)

        features["activity"] = activity
        features["dataset"] = file

        all_features.append(features)


feature_df = pd.DataFrame(all_features)


feature_cols = [
    col for col in feature_df.columns
    if col not in ["activity", "dataset"]
]

feature_df = feature_df[
    ["activity", "dataset"] + feature_cols
]

print("Shape of feature dataset:", feature_df.shape)
feature_df

Shape of feature dataset: (88, 44)


,activity,dataset,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,bending1,dataset1.csv,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
1,bending1,dataset2.csv,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
2,bending1,dataset3.csv,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
3,bending1,dataset4.csv,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
4,bending1,dataset5.csv,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,walking,dataset11.csv,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
84,walking,dataset12.csv,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
85,walking,dataset13.csv,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
86,walking,dataset14.csv,23.50,46.25,34.873229,35.25,4.531720,31.75,38.2500,0.0,...,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000


### 1(c)(iii) Standard Deviations and 90% Bootstrap Confidence Intervals

The standard deviation of each of the 42 extracted time-domain features is estimated across all 88 instances. A bootstrap procedure is then used to construct a 90% confidence interval for each standard deviation.

In [13]:
# Standard deviations and 90% bootstrap confidence intervals

np.random.seed(42)

feature_names = [
    col for col in feature_df.columns
    if col not in ["activity", "dataset"]
]

results = []

n_bootstrap = 10000

for feature in feature_names:
    values = feature_df[feature].dropna().to_numpy()

    # Estimated standard deviation
    std_estimate = np.std(values, ddof=1)

    # Bootstrap standard deviations
    bootstrap_stds = []

    for _ in range(n_bootstrap):
        sample = np.random.choice(
            values,
            size=len(values),
            replace=True
        )
        bootstrap_stds.append(np.std(sample, ddof=1))

    # 90% confidence interval
    lower = np.percentile(bootstrap_stds, 5)
    upper = np.percentile(bootstrap_stds, 95)

    results.append({
        "Feature": feature,
        "Standard Deviation": std_estimate,
        "90% CI Lower": lower,
        "90% CI Upper": upper
    })

bootstrap_results = pd.DataFrame(results)

bootstrap_results

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.624011,8.320129,10.800303
1,max1,4.207745,3.128908,5.119827
2,mean1,5.276431,4.645568,5.835215
3,median1,5.386624,4.730244,5.951775
4,std1,1.771282,1.567431,1.950774
5,1st_quart1,6.127846,5.530635,6.630072
6,3rd_quart1,5.031028,4.190767,5.757758
7,min2,0.000000,0.000000,0.000000
8,max2,5.059656,4.614543,5.393591
9,mean2,1.577908,1.401053,1.710460


### 1(c)(iv) Selection of the Three Most Important Time-Domain Features

I selected *minimum, mean, and maximum* as the three most important time-domain features.

- *Minimum:* captures the lowest observed value of a time series and helps describe the lower range of the activity signal.
- *Mean:* represents the overall average level of the signal and provides a useful measure of central tendency.
- *Maximum:* captures the highest observed value and helps describe the upper range of the activity signal.

Together, these three features provide a simple summary of the lower bound, central tendency, and upper bound of each time series. They can therefore help distinguish between activities with different signal levels and ranges.

## 2. ISLR 3.7.4

We have 100 observations with one predictor X and a quantitative response Y. We fit both a linear regression model and a cubic regression model.

### (a)

If the true relationship between X and Y is linear, I would expect the *cubic regression to have a lower or equal training RSS* than the linear regression. This is because the cubic model is more flexible and includes the linear model as a special case. Adding the X² and X³ terms cannot make the training fit worse.

### (b)

For the *test RSS, I would expect the **linear regression to have a lower test RSS*. Since the true relationship is linear, the linear model already fits the correct form of the relationship. The extra X² and X³ terms in the cubic model are unnecessary and can make the model more sensitive to the training data.

### (c)

If the true relationship is not linear, I would still expect the *cubic regression to have a lower or equal training RSS*. The cubic model is more flexible than the linear model, so it can fit the training observations at least as well as the linear model.

### (d)

For the *test RSS*, there is not enough information to say which model will be better. If the true relationship is strongly nonlinear, the cubic model may perform better because it can capture some of that curvature. However, if the relationship is only slightly nonlinear, the simpler linear model may perform better on new data. Therefore, the answer depends on the actual relationship between X and Y.